# R03 — Independent 0.5B Masked-Diffusion Replication

**RepairScopeX · Q1 evidence-strengthening batch (24 Oct 2026)**

Standalone notebook. Checkpointed and isolated. Never alters E10, E11 or V2-14.

## Aim and constraints
Independent masked-diffusion architecture (0.5B dLLM). Uses the **same deterministic 96-question external manifest** as R02 and outputs results on its own isolated Drive path. The model was adapted for code generation, so poor arithmetic performance is possible and must be reported.

Three interventions per question (288 calls), with 64 diffusion steps per call. These results do **not** represent post-hoc semantic remasking. The model card describes a mask-filled denoising decoder; this notebook implements that same mechanism and records actual forward evaluations.

**Colab compatibility patch:** This version removes an upstream model file's demonstration-only `dllm` import from a local copy while keeping the weights and inference implementation unchanged. It also distributes 64 denoising steps across three 32-token blocks correctly. Source revisions and hashes are archived with checkpoint records.

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
subprocess.run([sys.executable,'-m','pip','install','-q','datasets>=3.6,<5','huggingface-hub>=0.34,<1','pandas>=2','scipy>=1.12','accelerate>=1.10'],check=True)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import torch, torch.nn.functional as F, gc
assert torch.cuda.is_available(),'GPU runtime required'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','safetensors>=0.6'],check=True)
STUDY='RSX_Q1_R03_Independent_0p5B'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/Q1_R03_Independent_0p5B')
ROOT.mkdir(parents=True,exist_ok=True); RECS=ROOT/'records'; RECS.mkdir(exist_ok=True)


In [ ]:
import os, re, json, hashlib, random, time, math, zipfile, traceback, gc, csv
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from scipy.stats import binomtest
from google.colab import drive, files
from datasets import load_dataset
from huggingface_hub import repo_info

def sha(v):
    return hashlib.sha256(json.dumps(v,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def atomic_json(p,d):
    if p.exists():
        assert json.loads(p.read_text()) == d, f'PROTOCOL INTEGRITY MISMATCH {p}'
    else:
        q=p.with_suffix('.writing');q.write_text(json.dumps(d,sort_keys=True,indent=2,ensure_ascii=False));os.replace(q,p)
def save_record(d):
    assert d['protocol_hash']==PROTOCOL_HASH
    p=RECS/(hashlib.sha256(d['trial_id'].encode()).hexdigest()+'.json')
    atomic_json(p,d)
def stored(tid):
    p=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not p.exists():return None
    d=json.loads(p.read_text());assert d['trial_id']==tid and d['protocol_hash']==PROTOCOL_HASH
    return d
def all_records():
    out=[json.loads(p.read_text()) for p in RECS.glob('*.json')]
    assert len({r['trial_id'] for r in out})==len(out)
    assert all(x['protocol_hash']==PROTOCOL_HASH for x in out)
    return out

def norm_num(s):
    x=str(s).replace(',','').replace('$','').strip()
    try:return float(x)
    except: return None

def grade(output,target):
    # Three outcomes. Missing/incomplete/contradictory final answers are never counted as definitely wrong.
    text=str(output).strip(); t=float(target)
    forms=[]
    for pat in [r'(?im)^\s*FINAL_ANSWER\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*[.!]?\s*$',
                r'\\boxed\s*\{\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}',
                r'(?im)^\s*(?:final )?answer\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*[.!]?\s*$']:
        forms += [norm_num(m) for m in re.findall(pat,text) if norm_num(m) is not None]
    if len(forms)==0:
        return {'label':'ambiguous','value':None,'confidence':'missing_or_unfinished_final'}
    if any(not math.isclose(x,forms[-1],rel_tol=1e-8,abs_tol=1e-8) for x in forms):
        return {'label':'ambiguous','value':None,'confidence':'contradictory_final_markers'}
    return {'label':'correct' if math.isclose(forms[-1],t,rel_tol=1e-8,abs_tol=1e-8) else 'definite_wrong','value':forms[-1],'confidence':'explicit_final'}

def export_zip():
    path=Path('/content')/(STUDY+'_Snapshot.zip')
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if p.is_file() and not p.name.endswith('.writing'):z.write(p,p.relative_to(ROOT))
    print('Saved',path, 'size',path.stat().st_size)
    try: files.download(str(path))
    except Exception as e:print('Manual download possible:',path,repr(e))
    return path

# Public external test sets, selected before examining model outputs.
# Never substitute synthetic or training rows when download fails.
raw=[]; dataset_revs={}
for dsid,config in [('openai/gsm8k','main'),('ChilleD/SVAMP',None)]:
    prior_manifest_path=ROOT/'frozen_dataset_manifest.json'
    prior_manifest=json.loads(prior_manifest_path.read_text()) if prior_manifest_path.exists() else None
    rev=prior_manifest['dataset_revisions'][dsid] if prior_manifest else repo_info(dsid,repo_type='dataset').sha
    dataset_revs[dsid]=rev
    ds=load_dataset(dsid,config,split='test',revision=rev)
    temp=[]
    for n,r in enumerate(ds):
        if dsid=='openai/gsm8k':
            question=str(r['question']); answer=str(r['answer'])
            match=re.search(r'####\s*([+-]?[\d,]+(?:\.\d+)?)',answer)
            target=norm_num(match.group(1)) if match else None
        else:
            question=(str(r.get('Body',''))+' '+str(r.get('Question',''))).strip()
            target=norm_num(r.get('Answer'))
        if target is None or not question:continue
        ident='gsm8k' if 'gsm8k' in dsid else 'svamp'
        temp.append({'id':ident+':'+hashlib.sha256(question.encode()).hexdigest()[:16],
                     'dataset':ident,'prompt':question,'target':target,'original_index':n})
    chosen=sorted(temp,key=lambda r:sha(r['id']))[:48]
    assert len(chosen)==48,f'Insufficient {dsid} test items'
    raw.extend(chosen)
assert len(raw)==96 and len({x['id'] for x in raw})==96
# Store only public benchmark questions and answers (no chain of thought) and the exact dataset revisions.
manifest={'schema':'Q1-EXT-1','dataset_revisions':dataset_revs,'cases':raw,
          'selection':'48 minimum-SHA256 test question identifiers per dataset'}
atomic_json(ROOT/'frozen_dataset_manifest.json',manifest)
CASE_BY_ID={c['id']:c for c in raw}
print('Frozen datasets:',dataset_revs,'cases=',len(raw),'manifest=',sha(manifest))


In [ ]:
# R03 compatibility fix — reproducible model load without full dLLM framework.
# The released modeling_qwen2.py imports `dllm` only in its `if __name__ == '__main__'`
# demonstration block. Hugging Face's import checker nevertheless treats that
# demonstration-only dependency as required. We remove ONLY that block in a
# local copy of the pinned model snapshot. Core weights/model implementation
# are not modified. This avoids installing unrelated `dllm` packages or a
# heavyweight training framework in free Colab.
import ast, shutil, importlib, hashlib, pathlib, re
from huggingface_hub import snapshot_download

MODEL_ID = 'dllm-hub/Qwen2.5-Coder-0.5B-Instruct-diffusion-mdlm-v0.1'
MODEL_REV = (json.loads((ROOT/'protocol.json').read_text())['protocol']['model_revision']
             if (ROOT/'protocol.json').exists() else repo_info(MODEL_ID).sha)

# Download once into the standard HF cache; subsequent runs reuse cached bytes.
source_snapshot = pathlib.Path(snapshot_download(
    MODEL_ID,
    revision=MODEL_REV,
    allow_patterns=[
        '*.json', '*.jinja', '*.py', '*.safetensors',
        'merges.txt', 'vocab.json', '*.model', 'added_tokens.json'
    ],
))
origin = source_snapshot/'modeling_qwen2.py'
if not origin.exists():
    raise RuntimeError('Pinned model revision lacks modeling_qwen2.py')
original_source = origin.read_text(encoding='utf-8')
tree = ast.parse(original_source)

def is_demo_guard(node):
    if not isinstance(node, ast.If):
        return False
    t = node.test
    if not isinstance(t, ast.Compare) or len(t.ops)!=1 or not isinstance(t.ops[0],ast.Eq):
        return False
    def val(n):
        return n.id if isinstance(n,ast.Name) else (n.value if isinstance(n,ast.Constant) else None)
    return {val(t.left),val(t.comparators[0])} == {'__name__','__main__'}

guards = [node for node in tree.body if is_demo_guard(node)]
if len(guards)!=1:
    raise RuntimeError('Upstream model source changed; refusing undocumented modification.')
guard = guards[0]
removed = original_source.splitlines(keepends=True)[guard.lineno-1:guard.end_lineno]
if 'import dllm' not in ''.join(removed):
    raise RuntimeError('Expected demonstration-only dllm import not present. Verify new upstream code.')
source_lines = original_source.splitlines(keepends=True)
patched_source=''.join(source_lines[:guard.lineno-1] + source_lines[guard.end_lineno:])
if re.search(r'^\s*(?:from\s+dllm\b|import\s+dllm\b)',patched_source,re.M):
    raise RuntimeError('Runtime model implementation still imports dllm; cannot remove safely.')
compile(patched_source,'modeling_qwen2.py','exec')

local_dir=pathlib.Path('/content/rsx_r03_model_'+MODEL_REV[:12]);local_dir.mkdir(parents=True,exist_ok=True)
for p in source_snapshot.iterdir():
    if p.is_file() and p.name!='modeling_qwen2.py':
        q=local_dir/p.name
        if q.is_symlink():
            if q.resolve() != p.resolve():
                raise RuntimeError('Local model link points to unexpected source: '+str(q))
        elif q.exists():
            if hashlib.sha256(q.read_bytes()).digest()!=hashlib.sha256(p.read_bytes()).digest():
                raise RuntimeError('Local model cache collision: '+str(q))
        else:
            q.symlink_to(p.resolve())
patched_file=local_dir/'modeling_qwen2.py'
if patched_file.exists() and patched_file.read_text(encoding='utf-8')!=patched_source:
    raise RuntimeError('Refusing to overwrite mismatched local patched source')
patched_file.write_text(patched_source,encoding='utf-8')

provenance={
  'upstream_model_id':MODEL_ID,
  'upstream_revision':MODEL_REV,
  'upstream_source_sha256':hashlib.sha256(original_source.encode()).hexdigest(),
  'local_source_sha256':hashlib.sha256(patched_source.encode()).hexdigest(),
  'change':'Removed one __main__ standalone demonstration block that imports dllm; no model implementation changes',
  'model_weights':'Original pinned model weights, unmodified',
}
atomic_json(ROOT/'MODEL_SOURCE_PROVENANCE.json',provenance)
print('Model source compatibility check passed. Pinned revision:',MODEL_REV)

from transformers import AutoModelForMaskedLM, AutoTokenizer
DTYPE=torch.float16 if torch.cuda.get_device_capability(0)[0]<8 else torch.bfloat16
try:
    model=AutoModelForMaskedLM.from_pretrained(
        str(local_dir), trust_remote_code=True, torch_dtype=DTYPE,
    ).to('cuda').eval()
    tokenizer=AutoTokenizer.from_pretrained(str(local_dir), trust_remote_code=True)
except Exception as exc:
    raise RuntimeError(
        'Pinned 0.5B model failed to load after demonstration-import fix; '
        'see underlying exception. No R03 trial has been overwritten.'
    ) from exc
MASK=tokenizer.mask_token_id
if MASK is None: raise RuntimeError('Missing mask_token_id; cannot run masked diffusion.')
PAD=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else (tokenizer.eos_token_id if tokenizer.eos_token_id is not None else MASK)
SETTINGS={'steps':64,'block_size':32,'budget_source':64,'budget_interventions':96,
          'temperature':0.0,'cfg':0.0,'remasking':'low_confidence',
          'attention':'official remote model','model_dtype':str(DTYPE)}

@torch.inference_mode()
def masked_generate(question,phase,seed):
    torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    messages=[{'role':'system','content':'You are a careful assistant.'},{'role':'user','content':question}]
    ids=tokenizer.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,enable_thinking=False)
    ids=torch.tensor(ids,dtype=torch.long,device='cuda').unsqueeze(0)
    n_prompt=ids.size(1);n_new=64 if phase=='source' else 96
    x=torch.cat([ids,torch.full((1,n_new),MASK,device='cuda',dtype=torch.long)],dim=1)
    blocks=n_new//32
    assert n_new%32==0 and 64>=blocks
    steps_by_block=[64//blocks + int(b<64%blocks) for b in range(blocks)]
    assert sum(steps_by_block)==64
    start=time.monotonic();nfe=0
    for blk in range(blocks):
        lo=n_prompt+blk*32;hi=lo+32
        steps_per_block=steps_by_block[blk]
        quota=[32//steps_per_block+(int(i)<32%steps_per_block) for i in range(steps_per_block)]
        for step,k in enumerate(quota):
            if k==0:continue
            mask=(x==MASK);mask[:,:lo]=False;mask[:,hi:]=False
            if not mask.any():continue
            logits=model(x).logits; nfe+=1
            prediction=logits.argmax(-1)
            probs=F.softmax(logits.float(),dim=-1).gather(-1,prediction.unsqueeze(-1)).squeeze(-1)
            probs=probs.masked_fill(~mask,float('-inf'))
            # Select k highest-confidence currently masked tokens, preserving filled positions.
            top=probs.topk(min(k,int(mask.sum())),dim=-1).indices
            x[0,top[0]]=prediction[0,top[0]]
    out=tokenizer.decode(x[0,n_prompt:].tolist(),skip_special_tokens=True).strip()
    return {'output':out,'nfe':nfe,'wall_seconds':round(time.monotonic()-start,3),
            'generated_tokens':n_new,'stop_reason':'fixed_mask_budget'}

def prompt_for(c,phase,source=None):
    q=c['prompt']
    if phase=='source':return q+'\nSolve and end with FINAL_ANSWER: <number>.'
    if phase=='finalize':return q+'\nA previous response said:\n'+str(source)[:800]+'\nVerify and reply FINAL_ANSWER: <number>.'
    if phase=='regenerate':return q+'\nSolve independently from scratch and end with FINAL_ANSWER: <number>.'
    raise ValueError(phase)


**Model loading compatibility:** The released `modeling_qwen2.py` contains an optional `if __name__ == "__main__"` example that imports `dllm`. Hugging Face checks that dependency even though it is not executed in inference. This notebook copies the pinned snapshot into a local model path and strips only that demonstration block. The actual neural network code and weight files are untouched. No unrelated `pip install dllm` package is used.

In [ ]:
import pandas as pd
phase_names=('source','finalize','regenerate')
PLAN=[(c['id'],action) for c in raw for action in phase_names]
# Independently version the two architectures; never merge trial stores.
PROTOCOL={'study':STUDY,'manifest_sha256':sha(manifest),'phase_names':list(phase_names),
          'model_id':MODEL_ID,'model_revision':MODEL_REV,'config':SETTINGS,
          'expected_trials':len(PLAN),'planned_pairs':96}
PROTOCOL_HASH=sha(PROTOCOL)
atomic_json(ROOT/'protocol.json',{'protocol':PROTOCOL,'protocol_hash':PROTOCOL_HASH})
print('Study',STUDY,'protocol',PROTOCOL_HASH,'completed',len(all_records()),'/',len(PLAN))


In [ ]:
SESSION_NEW_LIMIT=10; SESSION_MINUTES=26
start=time.monotonic();new=0;error=None
for pid,phase in PLAN:
    if new>=SESSION_NEW_LIMIT or time.monotonic()-start>=SESSION_MINUTES*60:break
    tid=STUDY+'|'+pid+'|'+phase
    if stored(tid):continue
    c=CASE_BY_ID[pid];source=stored(STUDY+'|'+pid+'|source')
    if phase!='source' and source is None:raise RuntimeError('source not saved '+pid)
    try:
        seed=920000+int(hashlib.sha256(pid.encode()).hexdigest()[:7],16)%999999
        pred=masked_generate(prompt_for(c,phase,source['output'] if source else None),phase,seed)
        rec={'trial_id':tid,'protocol_hash':PROTOCOL_HASH,'case_id':pid,'dataset':c['dataset'],
             'phase':phase,'seed':seed,'question_sha256':sha(c['prompt']),'target':c['target'],
             'grade':grade(pred['output'],c['target']),**pred}
        save_record(rec);new+=1
        print('NEW',new,pid,phase,rec['grade']['label'],flush=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache();error='CUDA OOM';break
    except Exception as e:error=repr(e);break
print('Saved',len(all_records()),'/',len(PLAN),'new',new,'error',error)
if error:(ROOT/'last_error.json').write_text(json.dumps({'error':error,'problem':pid,'phase':phase},indent=2))


In [ ]:
import pandas as pd
records=all_records(); by={(r['case_id'],r['phase']):r for r in records}
complete=[c for c in raw if all((c['id'],a) in by for a in phase_names)]
rows=[]
for c in complete:
    row={'case_id':c['id'],'dataset':c['dataset']}
    for phase in phase_names:
        r=by[(c['id'],phase)]
        row[phase+'_correct']=int(r['grade']['label']=='correct')
        row[phase+'_ambiguous']=int(r['grade']['label']=='ambiguous')
        row[phase+'_time']=float(r['wall_seconds'])
        row[phase+'_passes']=float(r.get('nfe',0))
    for a in ['finalize','regenerate']:
        row[a+'_recovery']=int(not row['source_correct'] and row[a+'_correct'])
        row[a+'_damage']=int(row['source_correct'] and not row[a+'_correct'])
    rows.append(row)
df=pd.DataFrame(rows)
if not df.empty:
    df.to_csv(ROOT/'paired_results.csv',index=False)
    print('COMPLETED PAIRED PROBLEMS',len(df), '/',len(raw))
    for phase in phase_names:
        print(phase,'accuracy',round(df[phase+'_correct'].mean(),4),'ambiguous',int(df[phase+'_ambiguous'].sum()),
              'median wall sec',round(df[phase+'_time'].median(),3),'mean NFE',round(df[phase+'_passes'].mean(),2))
    for a in ['finalize','regenerate']:
        wins=int(((df[a+'_correct']==1)&(df['source_correct']==0)).sum())
        losses=int(((df[a+'_correct']==0)&(df['source_correct']==1)).sum())
        p=float(binomtest(min(wins,losses),wins+losses,0.5).pvalue) if wins+losses else 1.0
        print('vs preserve',a,'recovery',wins,'damage',losses,'paired exact p',p)
else:print('No complete case triples yet')
export_zip()
